# 28. Equality, hashing, and operator overloading

**Level:** Intermediate to advanced

**Learning outcome:** Implement value semantics and natural unit-safe arithmetic without confusing identity with equality.

**Prerequisites:** 11-12: dataclasses, value objects, and special methods.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](27_Integrated_Network_Automation_Capstones.ipynb) | [Next notebook](29_Function_Decorators_and_Method_Binding.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
A session often has identity; a route key often has value. Two independently parsed route keys
can mean the same destination and VRF even though they are not the same object. `is` asks about
identity. `==` asks for an equality decision. Choose that decision from the domain, not from
whichever fields happen to be convenient.

A custom `__eq__` should normally return NotImplemented for unsupported other types. That sentinel
allows Python's comparison machinery to consider the other operand or a fallback; it is not
the exception NotImplementedError. Do not use the sentinel as a boolean.

A dictionary key must have a stable hash relationship: equal keys have equal hashes, and the
compared state must not change while the key is stored. A frozen dataclass containing only
immutable fields is often the simplest implementation. Hash values are not unique identifiers
and need not remain stable between processes. A read-only public property alone does not make
arbitrary internal state immutable.

Operators are public interfaces too. Addition of compatible traffic totals is natural; addition
of live sessions is not. Keep units in names, validate operands, return a new value, and reject
incompatible operations honestly. `__radd__` handles the reflected addition path; `sum` begins
with integer zero by default, so supporting that specific identity is a deliberate design choice.

## Worked example 1: Manual equality without accidental cross-type matches

**Problem:** Treat two normalized device identity values as equal while keeping unrelated types distinct.

### Step-by-step explanation
1. Normalize at construction so case and surrounding whitespace follow one identity policy.
2. Use an exact-type check to avoid silently promising subclass equality semantics.
3. Return NotImplemented for a string rather than pretending it is an identity object.
4. Demonstrate that defining equality without a hash leaves this mutable model unhashable.

**Expected output:**
```text
mutable equality model is intentionally unhashable
True False
```

**Common mistake / limitation:** Hostname-only identity is a local policy. Real inventories may require tenant, site, or VRF scope.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
class DeviceIdentity:
    def __init__(self, hostname):
        self.hostname = hostname.strip().lower()

    def __eq__(self, other):
        if type(other) is not type(self):
            return NotImplemented
        return self.hostname == other.hostname

first = DeviceIdentity(" EDGE-01 ")
second = DeviceIdentity("edge-01")
assert first == second
assert first is not second
assert first.__eq__("edge-01") is NotImplemented
assert (first == "edge-01") is False
try:
    hash(first)
except TypeError:
    print("mutable equality model is intentionally unhashable")
else:
    raise AssertionError("unexpected hashability")
print(first == second, first is second)

mutable equality model is intentionally unhashable
True False


### Practice 1

Add a tenant field and prove same-hostname values in different tenants compare unequal.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: Immutable scoped route keys work as dictionary keys

**Problem:** Use canonical prefixes and VRF scope to deduplicate equivalent logical keys.

### Step-by-step explanation
1. Parse with strict=False to intentionally normalize host bits to a network boundary.
2. Keep both VRF and network in frozen fields so equality includes the allocation scope.
3. Construct a separately parsed equal key and use it for dictionary lookup.
4. Do not print numeric hashes because they are not a persistent identifier.

**Expected output:**
```text
next-hop-A next-hop-B
```

**Common mistake / limitation:** Type hints do not enforce constructor input types; production should control all construction paths.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
from dataclasses import dataclass
from ipaddress import IPv4Network, ip_network

@dataclass(frozen=True)
class RouteKey:
    vrf: str
    network: IPv4Network

    @classmethod
    def parse(cls, vrf, prefix):
        network = ip_network(prefix, strict=False)
        if network.version != 4:
            raise ValueError("this lesson requires IPv4")
        return cls(vrf.strip().lower(), network)

first = RouteKey.parse(" BLUE ", "192.0.2.7/24")
same = RouteKey.parse("blue", "192.0.2.0/24")
other_scope = RouteKey.parse("red", "192.0.2.0/24")
routes = {first: "next-hop-A", other_scope: "next-hop-B"}
assert first == same and hash(first) == hash(same)
assert routes[same] == "next-hop-A"
assert len(routes) == 2
print(routes[same], routes[other_scope])

next-hop-A next-hop-B


### Practice 2

Support IPv6 with an explicit union and add tests for equivalent IPv6 prefix spellings.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: Natural addition of byte totals

**Problem:** Add compatible immutable measurements and support sum without accepting arbitrary numeric units.

### Step-by-step explanation
1. Validate that the measurement represents a nonnegative integer byte count.
2. Return a new ByteTotal when both operands have compatible units.
3. Accept only the exact integer-zero identity for reflected addition used by sum.
4. Check immutability of the operands and explicit rejection of an untyped integer.

**Expected output:**
```text
ambiguous untyped unit rejected
bytes: 3000
```

**Common mistake / limitation:** This represents a total, not a rate. Do not add bytes and bits-per-second as though units matched.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ByteTotal:
    value: int

    def __post_init__(self):
        if type(self.value) is not int or self.value < 0:
            raise ValueError("nonnegative integer bytes required")

    def __add__(self, other):
        if not isinstance(other, ByteTotal):
            return NotImplemented
        return ByteTotal(self.value + other.value)

    def __radd__(self, other):
        if type(other) is int and other == 0:
            return self
        return NotImplemented

first, second = ByteTotal(1000), ByteTotal(2000)
total = sum([first, second])
assert total == ByteTotal(3000)
assert first.value == 1000
try:
    first + 5
except TypeError:
    print("ambiguous untyped unit rejected")
else:
    raise AssertionError("untyped addition accepted")
print("bytes:", total.value)

ambiguous untyped unit rejected
bytes: 3000


### Practice 3

Add a named to_bits() conversion returning a separate BitTotal type.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: Equality, hashing, and operator overloading

**You should now be able to:** Implement value semantics and natural unit-safe arithmetic without confusing identity with equality.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/reference/datamodel.html`
- `https://docs.python.org/3.13/library/dataclasses.html`

References describe Python contracts. They do not certify any vendor command or network policy.